In [27]:
!pip install pyspark

In [28]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    sum as spark_sum,
    round as spark_round,
    desc
)

In [29]:
spark = SparkSession.builder \
    .appName("GoIT_DE_HW_03") \
    .getOrCreate()

In [30]:
import os

# Автоматичний пошук CSV-файлів
files_list = os.listdir('/content')

print("Знайдені файли:", files_list)

# Завантаження у Spark DataFrame
users_df = spark.read.csv(
    '/content/users.csv',
    header=True,
    inferSchema=True
)

purchases_df = spark.read.csv(
    '/content/purchases.csv',
    header=True,
    inferSchema=True
)

products_df = spark.read.csv(
    '/content/products.csv',
    header=True,
    inferSchema=True
)

# Перевірка
users_df.show()
purchases_df.show()
products_df.show()

Знайдені файли: ['.config', 'sample_data', 'products.csv', '.ipynb_checkpoints', 'users.csv', 'purchases.csv', 'drive']
+-------+-------+---+------------------+
|user_id|   name|age|             email|
+-------+-------+---+------------------+
|      1| User_1| 45| user1@example.com|
|      2| User_2| 48| user2@example.com|
|      3| User_3| 36| user3@example.com|
|      4| User_4| 46| user4@example.com|
|      5| User_5| 29| user5@example.com|
|      6| User_6| 39| user6@example.com|
|      7| User_7| 24| user7@example.com|
|      8| User_8| 44| user8@example.com|
|      9| User_9| 27| user9@example.com|
|     10|User_10| 43|user10@example.com|
|     11|User_11| 36|user11@example.com|
|     12|User_12| 24|user12@example.com|
|     13|User_13| 47|user13@example.com|
|     14|User_14| 20|user14@example.com|
|     15|User_15| 30|user15@example.com|
|     16|User_16| 30|user16@example.com|
|     17|User_17| 35|user17@example.com|
|     18|User_18| 45|user18@example.com|
|     19|User_19| 4

In [31]:
users_df = spark.read.csv(
    "users.csv",
    header=True,
    inferSchema=True
)

purchases_df = spark.read.csv(
    "purchases.csv",
    header=True,
    inferSchema=True
)

products_df = spark.read.csv(
    "products.csv",
    header=True,
    inferSchema=True
)

In [32]:
users_df.show()
purchases_df.show()
products_df.show()

+-------+-------+---+------------------+
|user_id|   name|age|             email|
+-------+-------+---+------------------+
|      1| User_1| 45| user1@example.com|
|      2| User_2| 48| user2@example.com|
|      3| User_3| 36| user3@example.com|
|      4| User_4| 46| user4@example.com|
|      5| User_5| 29| user5@example.com|
|      6| User_6| 39| user6@example.com|
|      7| User_7| 24| user7@example.com|
|      8| User_8| 44| user8@example.com|
|      9| User_9| 27| user9@example.com|
|     10|User_10| 43|user10@example.com|
|     11|User_11| 36|user11@example.com|
|     12|User_12| 24|user12@example.com|
|     13|User_13| 47|user13@example.com|
|     14|User_14| 20|user14@example.com|
|     15|User_15| 30|user15@example.com|
|     16|User_16| 30|user16@example.com|
|     17|User_17| 35|user17@example.com|
|     18|User_18| 45|user18@example.com|
|     19|User_19| 47|user19@example.com|
|     20|User_20| 43|user20@example.com|
+-------+-------+---+------------------+
only showing top

In [33]:
users_df = users_df.dropna()
purchases_df = purchases_df.dropna()
products_df = products_df.dropna()

In [34]:
users_df.show()
purchases_df.show()
products_df.show()

+-------+-------+---+------------------+
|user_id|   name|age|             email|
+-------+-------+---+------------------+
|      1| User_1| 45| user1@example.com|
|      2| User_2| 48| user2@example.com|
|      3| User_3| 36| user3@example.com|
|      4| User_4| 46| user4@example.com|
|      5| User_5| 29| user5@example.com|
|      6| User_6| 39| user6@example.com|
|      7| User_7| 24| user7@example.com|
|      8| User_8| 44| user8@example.com|
|      9| User_9| 27| user9@example.com|
|     10|User_10| 43|user10@example.com|
|     11|User_11| 36|user11@example.com|
|     12|User_12| 24|user12@example.com|
|     13|User_13| 47|user13@example.com|
|     14|User_14| 20|user14@example.com|
|     15|User_15| 30|user15@example.com|
|     16|User_16| 30|user16@example.com|
|     17|User_17| 35|user17@example.com|
|     18|User_18| 45|user18@example.com|
|     19|User_19| 47|user19@example.com|
|     20|User_20| 43|user20@example.com|
+-------+-------+---+------------------+
only showing top

In [35]:
total_purchases = purchases_df.join(
    products_df,
    on="product_id",
    how="inner"
)

total_by_category = total_purchases.groupBy("category").agg(
    spark_sum(col("quantity") * col("price")).alias("total_amount")
)

total_by_category.show()

+-----------+------------------+
|   category|      total_amount|
+-----------+------------------+
|       Home|1523.4999999999998|
|     Sports|1802.4999999999998|
|Electronics|1174.7999999999997|
|   Clothing|             790.3|
|     Beauty| 459.8999999999999|
+-----------+------------------+



In [36]:
users_purchases = purchases_df.join(
    users_df,
    on="user_id",
    how="inner"
)

young_users = users_purchases.filter(
    (col("age") >= 18) & (col("age") <= 25)
)

young_users_purchases = young_users.join(
    products_df,
    on="product_id",
    how="inner"
)

young_by_category = young_users_purchases.groupBy("category").agg(
    spark_sum(col("quantity") * col("price")).alias("total_amount")
)

young_by_category.show()

+-----------+------------------+
|   category|      total_amount|
+-----------+------------------+
|       Home|             361.1|
|     Sports|310.49999999999994|
|Electronics|             249.6|
|   Clothing|             245.0|
|     Beauty|41.400000000000006|
+-----------+------------------+



In [37]:
from pyspark.sql.window import Window

window_spec = Window.partitionBy()

young_with_percentage = young_by_category.withColumn(
    "percentage",
    spark_round(
        col("total_amount") /
        spark_sum("total_amount").over(window_spec) * 100,
        2
    )
)

young_with_percentage.show()

+-----------+------------------+----------+
|   category|      total_amount|percentage|
+-----------+------------------+----------+
|       Home|             361.1|      29.9|
|     Sports|310.49999999999994|     25.71|
|Electronics|             249.6|     20.67|
|   Clothing|             245.0|     20.29|
|     Beauty|41.400000000000006|      3.43|
+-----------+------------------+----------+



In [38]:
top_3_categories = young_with_percentage.orderBy(
    desc("percentage")
).limit(3)

top_3_categories.show()

+-----------+------------------+----------+
|   category|      total_amount|percentage|
+-----------+------------------+----------+
|       Home|             361.1|      29.9|
|     Sports|310.49999999999994|     25.71|
|Electronics|             249.6|     20.67|
+-----------+------------------+----------+

